# RQ3 Pre-model Diagnostics

This notebook prepares pre-model diagnostics for RQ3 social determinants models. It does **not** estimate the final explanatory models. It checks missingness, variable distributions, correlations, multicollinearity, country fixed-effect identification, and spatial residual autocorrelation so that the formal RQ3 models can be specified transparently.

In [ ]:

from pathlib import Path
import json
import sys
import warnings

PROJECT = Path('/Users/nico/Research/Energy sufficiency')
LOCAL_PKG = PROJECT / 'Process/python_packages'
if LOCAL_PKG.exists() and str(LOCAL_PKG) not in sys.path:
    sys.path.insert(0, str(LOCAL_PKG))

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_theme(style='whitegrid', font='DejaVu Sans')

DATA_PATH = PROJECT / 'Data/global_SE_EC_sufficiency_controls_2015.csv'
RESULTS_DIR = PROJECT / 'Results/RQ3_social_determinants'
SI_DIR = RESULTS_DIR / 'SI'
FIG_DIR = RESULTS_DIR / 'Figures'
TABLE_DIR = RESULTS_DIR / 'Tables'
DIAG_DIR = RESULTS_DIR / 'Model_diagnostics'
PROCESS_DIR = PROJECT / 'Process/RQ3_pretest'
for d in [RESULTS_DIR, SI_DIR, FIG_DIR, TABLE_DIR, DIAG_DIR, PROCESS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT)
print('Data:', DATA_PATH)
print('Results:', RESULTS_DIR)


In [ ]:

df_raw = pd.read_csv(DATA_PATH)
print('Raw shape:', df_raw.shape)
print('Unique GDL:', df_raw['GDLcode'].nunique())
print('Countries:', df_raw['iso_code'].nunique())
print('Analytical regions:', df_raw['analytical_region'].nunique())
df_raw.head()


In [ ]:

def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if m.sum() == 0:
        return np.nan
    return np.average(x[m], weights=w[m])

def clean_model_data(df):
    d = df.copy()
    for c in ['income', 'wealth']:
        d[c + '_clean'] = d[c].where(d[c] >= 0, np.nan)
    d['SES_raw'] = d['income_clean'].combine_first(d['wealth_clean'])
    d['female_share'] = d['gender']
    d['log_sufficiency_ratio'] = np.log(d['sufficiency_ratio'].where(d['sufficiency_ratio'] > 0))
    d['log_EC2015_pc'] = np.log(d['EC2015_pc'].where(d['EC2015_pc'] > 0))
    d['log_GDP'] = np.log(d['GDP'].where(d['GDP'] > 0))
    d['GDP_pc'] = d['GDP'] / d['population'].replace(0, np.nan)
    d['log_GDP_pc'] = np.log(d['GDP_pc'].where(d['GDP_pc'] > 0))
    d['log_population'] = np.log(d['population'].where(d['population'] > 0))
    return d

df = clean_model_data(df_raw)

core_social = ['SES_raw', 'rural', 'education', 'female_share', 'age', 'Hhsize']
controls_candidate = ['log_GDP_pc', 'log_population', 'total_HDD', 'total_CDD', 'HDD_days', 'CDD_days', 'mean_temp']
recommended_controls = ['log_GDP_pc', 'log_population', 'total_HDD', 'total_CDD']
outcomes = ['log_sufficiency_ratio', 'sufficiency_ratio', 'sufficiency_gap', 'sufficiency_deficit']

model_vars = ['log_sufficiency_ratio'] + core_social + recommended_controls + ['iso_code', 'GDLcode', 'analytical_region', 'population']
model_sample = df.dropna(subset=['log_sufficiency_ratio'] + core_social + recommended_controls + ['iso_code']).copy()
model_sample = model_sample[np.isfinite(model_sample['log_sufficiency_ratio']) & (model_sample['population'] > 0)].copy()

print('Clean variables added:', ['SES_raw', 'female_share', 'log_sufficiency_ratio', 'log_GDP_pc', 'log_population'])
print('Model sample rows:', len(model_sample))
print('Model sample countries:', model_sample['iso_code'].nunique())
print('Model sample regions:', model_sample['analytical_region'].nunique())


In [ ]:

# Missingness table for all key variables.
key_vars = [
    'GDLcode', 'iso_code', 'country', 'gdl_region', 'analytical_region',
    'population', 'GDP', 'GDP_pc', 'log_GDP_pc',
    'age', 'education', 'gender', 'female_share', 'Hhsize', 'income', 'wealth', 'SES_raw', 'rural',
    'EC2015_pc', 'sufficiency_ratio', 'log_sufficiency_ratio', 'sufficiency_gap', 'sufficiency_deficit', 'sufficiency_excess',
    'total_HDD', 'total_CDD', 'HDD_days', 'CDD_days', 'mean_temp'
]
missing = []
for col in key_vars:
    if col not in df.columns:
        continue
    missing.append({
        'variable': col,
        'n_missing': int(df[col].isna().sum()),
        'pct_missing': float(df[col].isna().mean()),
        'n_nonmissing': int(df[col].notna().sum()),
    })
missing_df = pd.DataFrame(missing)
missing_df.to_csv(PROCESS_DIR / 'rq3_pretest_missingness_all_key_variables.csv', index=False)
missing_df.to_csv(SI_DIR / 'Table_SI_RQ3_missingness_key_variables.csv', index=False)
missing_df


In [ ]:

# Sample inclusion diagnostics.
inclusion_rows = []
steps = [
    ('Raw GDL rows', df),
    ('Valid population > 0', df[df['population'] > 0]),
    ('Valid positive sufficiency_ratio', df[df['log_sufficiency_ratio'].notna()]),
    ('Valid social predictors', df.dropna(subset=core_social)),
    ('Valid recommended controls', df.dropna(subset=recommended_controls)),
    ('Complete pretest model sample', model_sample),
]
for label, dat in steps:
    inclusion_rows.append({
        'step': label,
        'n_gdl': int(len(dat)),
        'n_countries': int(dat['iso_code'].nunique(dropna=True)) if 'iso_code' in dat else np.nan,
        'n_regions': int(dat['analytical_region'].nunique(dropna=True)) if 'analytical_region' in dat else np.nan,
        'population_sum': float(dat['population'].sum(skipna=True)) if 'population' in dat else np.nan,
    })
inclusion = pd.DataFrame(inclusion_rows)
inclusion.to_csv(PROCESS_DIR / 'rq3_pretest_sample_inclusion.csv', index=False)
inclusion.to_csv(SI_DIR / 'Table_SI_RQ3_sample_inclusion.csv', index=False)
inclusion


In [ ]:

# Descriptive statistics and distributions. These are process outputs, not necessarily SI figures.
describe_vars = outcomes + core_social + recommended_controls + ['HDD_days', 'CDD_days', 'mean_temp', 'GDP', 'population']
desc = df[describe_vars].describe(percentiles=[.01,.05,.10,.25,.50,.75,.90,.95,.99]).T
desc.to_csv(PROCESS_DIR / 'rq3_pretest_descriptive_statistics.csv')

plot_vars = ['log_sufficiency_ratio'] + core_social + recommended_controls
n = len(plot_vars)
fig, axes = plt.subplots(3, 4, figsize=(13, 8), constrained_layout=True)
axes = axes.ravel()
for ax, var in zip(axes, plot_vars):
    sns.histplot(model_sample[var], kde=True, ax=ax, color='#4c78a8', bins=30)
    ax.set_title(var, fontsize=10)
    ax.set_xlabel('')
for ax in axes[len(plot_vars):]:
    ax.axis('off')
fig.suptitle('RQ3 pretest variable distributions in complete model sample', y=1.02, fontsize=13)
fig.savefig(PROCESS_DIR / 'rq3_pretest_variable_distributions.png', bbox_inches='tight')
plt.close(fig)
desc


In [ ]:

# Correlation matrices with significance stars.
def corr_with_pvalues(data, variables, method='pearson'):
    rows = []
    rmat = pd.DataFrame(np.nan, index=variables, columns=variables)
    pmat = pd.DataFrame(np.nan, index=variables, columns=variables)
    nmat = pd.DataFrame(0, index=variables, columns=variables)
    for i, a in enumerate(variables):
        for j, b in enumerate(variables):
            if a == b:
                x = data[[a]].dropna()
                n = len(x)
                nmat.loc[a, b] = n
                if n >= 1:
                    rmat.loc[a, b] = 1.0
                    pmat.loc[a, b] = 0.0
                continue
            x = data[[a, b]].dropna()
            n = len(x)
            nmat.loc[a, b] = n
            if n < 3:
                continue
            if method == 'pearson':
                r, p = stats.pearsonr(x[a], x[b])
            else:
                r, p = stats.spearmanr(x[a], x[b])
            rmat.loc[a, b] = float(r)
            pmat.loc[a, b] = float(p)
    return rmat, pmat, nmat

def stars(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''

def annot_from(rmat, pmat):
    annot = rmat.copy().astype(object)
    for i in rmat.index:
        for j in rmat.columns:
            val = rmat.loc[i, j]
            annot.loc[i, j] = '' if pd.isna(val) else f'{val:.2f}{stars(pmat.loc[i,j])}'
    return annot

corr_vars = ['log_sufficiency_ratio', 'sufficiency_gap', 'sufficiency_deficit'] + core_social + ['log_GDP_pc', 'log_population', 'total_HDD', 'total_CDD', 'HDD_days', 'CDD_days', 'mean_temp']
pearson_r, pearson_p, pearson_n = corr_with_pvalues(df, corr_vars, 'pearson')
spearman_r, spearman_p, spearman_n = corr_with_pvalues(df, corr_vars, 'spearman')

pearson_r.to_csv(SI_DIR / 'Table_SI_RQ3_correlation_pearson_r.csv')
pearson_p.to_csv(SI_DIR / 'Table_SI_RQ3_correlation_pearson_p.csv')
spearman_r.to_csv(SI_DIR / 'Table_SI_RQ3_correlation_spearman_r.csv')
spearman_p.to_csv(SI_DIR / 'Table_SI_RQ3_correlation_spearman_p.csv')

for name, rmat, pmat in [('pearson', pearson_r, pearson_p), ('spearman', spearman_r, spearman_p)]:
    fig, ax = plt.subplots(figsize=(12.8, 10.5), constrained_layout=True)
    sns.heatmap(
        rmat.astype(float), cmap='vlag', center=0, vmin=-1, vmax=1,
        annot=annot_from(rmat, pmat), fmt='', annot_kws={'fontsize': 7.5},
        linewidths=.35, linecolor='white', square=True,
        cbar_kws={'label': f'{name.title()} correlation'}, ax=ax
    )
    ax.set_title(f'RQ3 pre-model {name.title()} correlations with significance stars', fontsize=13, pad=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    fig.savefig(SI_DIR / f'Figure_SI_RQ3_correlation_heatmap_{name}.png', bbox_inches='tight')
    plt.close(fig)

# High-correlation pair list for modelling decisions.
pairs = []
for i, a in enumerate(corr_vars):
    for b in corr_vars[i+1:]:
        r = pearson_r.loc[a, b]
        p = pearson_p.loc[a, b]
        if pd.notna(r) and abs(r) >= 0.65:
            pairs.append({'var1': a, 'var2': b, 'pearson_r': float(r), 'p_value': float(p), 'stars': stars(p), 'abs_r': abs(float(r))})
high_corr = pd.DataFrame(pairs).sort_values('abs_r', ascending=False)
high_corr.to_csv(PROCESS_DIR / 'rq3_pretest_high_correlation_pairs_abs_ge_0_65.csv', index=False)
high_corr


In [ ]:

# VIF diagnostics. We intentionally test an over-specified climate set to reveal collinearity.
def compute_vif(data, variables):
    x = data[variables].replace([np.inf, -np.inf], np.nan).dropna().copy()
    out = []
    if len(x) < 5:
        return pd.DataFrame()
    x_std = (x - x.mean()) / x.std(ddof=0)
    x_std = x_std.replace([np.inf, -np.inf], np.nan).dropna()
    X = sm.add_constant(x_std, has_constant='add')
    for idx, var in enumerate(X.columns):
        if var == 'const':
            continue
        try:
            vif = variance_inflation_factor(X.values, idx)
        except Exception:
            vif = np.inf
        out.append({'variable': var, 'VIF': float(vif), 'n_complete': int(len(x_std)), 'spec_variables': ', '.join(variables)})
    return pd.DataFrame(out)

vif_specs = {
    'A_full_candidate_all_climate': core_social + ['log_GDP_pc', 'log_population', 'total_HDD', 'total_CDD', 'HDD_days', 'CDD_days', 'mean_temp'],
    'B_recommended_total_degree_days': core_social + ['log_GDP_pc', 'log_population', 'total_HDD', 'total_CDD'],
    'C_temperature_only_climate': core_social + ['log_GDP_pc', 'log_population', 'mean_temp'],
    'D_total_degree_days_plus_mean_temp': core_social + ['log_GDP_pc', 'log_population', 'total_HDD', 'total_CDD', 'mean_temp'],
}

vif_tables = []
for spec, vars_ in vif_specs.items():
    tab = compute_vif(df, vars_)
    tab.insert(0, 'specification', spec)
    vif_tables.append(tab)
vif_all = pd.concat(vif_tables, ignore_index=True)
vif_all.to_csv(SI_DIR / 'Table_SI_RQ3_VIF_diagnostics.csv', index=False)

recommended_vif = vif_all[vif_all['specification'] == 'B_recommended_total_degree_days'].copy().sort_values('VIF', ascending=True)
fig, ax = plt.subplots(figsize=(7.5, 5.8), constrained_layout=True)
colors = ['#d95f02' if v >= 10 else '#e6ab02' if v >= 5 else '#4c78a8' for v in recommended_vif['VIF']]
ax.barh(recommended_vif['variable'], recommended_vif['VIF'], color=colors)
ax.axvline(5, color='#777777', linestyle='--', linewidth=1, label='VIF = 5')
ax.axvline(10, color='#333333', linestyle=':', linewidth=1, label='VIF = 10')
ax.set_xlabel('Variance inflation factor')
ax.set_title('RQ3 VIF diagnostics for provisional main specification')
ax.legend(frameon=False)
fig.savefig(SI_DIR / 'Figure_SI_RQ3_VIF_recommended_specification.png', bbox_inches='tight')
plt.close(fig)

vif_all.sort_values(['specification', 'VIF'], ascending=[True, False])


In [ ]:

# Country fixed-effect identification diagnostics.
country_counts = model_sample.groupby(['iso_code', 'country', 'analytical_region'], dropna=False).agg(
    n_gdl=('GDLcode', 'count'),
    population=('population', 'sum'),
    mean_log_sufficiency_ratio=('log_sufficiency_ratio', 'mean'),
).reset_index()
country_counts['single_gdl_country'] = country_counts['n_gdl'] == 1
country_counts.to_csv(PROCESS_DIR / 'rq3_pretest_country_fe_sample_counts.csv', index=False)

within_rows = []
for var in core_social + recommended_controls:
    tmp = model_sample.groupby('iso_code')[var].agg(['count', 'mean', 'std']).reset_index()
    tmp['variable'] = var
    tmp['near_zero_within_sd'] = tmp['std'].fillna(0) < 1e-8
    within_rows.append(tmp[['iso_code', 'variable', 'count', 'mean', 'std', 'near_zero_within_sd']])
within_variation = pd.concat(within_rows, ignore_index=True)
within_variation.to_csv(PROCESS_DIR / 'rq3_pretest_within_country_variation.csv', index=False)
within_summary = within_variation.groupby('variable').agg(
    n_countries=('iso_code','count'),
    countries_near_zero_within_sd=('near_zero_within_sd','sum'),
    median_within_sd=('std','median'),
    p10_within_sd=('std', lambda s: np.nanquantile(s, .10)),
    p90_within_sd=('std', lambda s: np.nanquantile(s, .90)),
).reset_index()
within_summary.to_csv(PROCESS_DIR / 'rq3_pretest_within_country_variation_summary.csv', index=False)

region_counts = model_sample.groupby('analytical_region').agg(n_gdl=('GDLcode','count'), n_countries=('iso_code','nunique'), population=('population','sum')).reset_index()
region_counts.to_csv(PROCESS_DIR / 'rq3_pretest_region_model_sample_counts.csv', index=False)

print('Countries in complete model sample:', country_counts.shape[0])
print('Single-GDL countries:', int(country_counts['single_gdl_country'].sum()))
print('Countries with >=2 GDL:', int((country_counts['n_gdl'] >= 2).sum()))
within_summary


In [ ]:

# Preliminary country-FE model only for residual diagnostics, not final interpretation.
model_data = model_sample.copy()
for var in core_social + recommended_controls:
    mu = model_data[var].mean()
    sd = model_data[var].std(ddof=0)
    model_data['z_' + var] = (model_data[var] - mu) / sd

rhs = ' + '.join(['z_' + v for v in core_social + recommended_controls]) + ' + C(iso_code)'
formula = 'log_sufficiency_ratio ~ ' + rhs
fe_model = smf.ols(formula, data=model_data).fit(cov_type='cluster', cov_kwds={'groups': model_data['iso_code']})
model_data['fe_residual'] = fe_model.resid
model_data[['GDLcode', 'iso_code', 'country', 'analytical_region', 'log_sufficiency_ratio', 'fe_residual']].to_csv(PROCESS_DIR / 'rq3_pretest_baseline_fe_residuals.csv', index=False)

resid_desc = pd.DataFrame({
    'metric': ['n_obs', 'n_countries', 'r_squared', 'adj_r_squared', 'residual_mean', 'residual_sd', 'residual_skew', 'residual_kurtosis'],
    'value': [
        fe_model.nobs, model_data['iso_code'].nunique(), fe_model.rsquared, fe_model.rsquared_adj,
        model_data['fe_residual'].mean(), model_data['fe_residual'].std(ddof=1),
        stats.skew(model_data['fe_residual'], nan_policy='omit'), stats.kurtosis(model_data['fe_residual'], nan_policy='omit')
    ]
})
resid_desc.to_csv(PROCESS_DIR / 'rq3_pretest_baseline_fe_residual_summary.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
sns.histplot(model_data['fe_residual'], kde=True, ax=axes[0], color='#4c78a8', bins=40)
axes[0].set_title('Baseline FE residual distribution')
stats.probplot(model_data['fe_residual'], dist='norm', plot=axes[1])
axes[1].set_title('Baseline FE residual Q-Q plot')
fig.savefig(PROCESS_DIR / 'rq3_pretest_baseline_fe_residual_diagnostics.png', bbox_inches='tight')
plt.close(fig)

resid_desc


In [ ]:

# Spatial residual diagnostics with libpysal/esda/spreg.
spatial_result = {'status': 'not_run'}
try:
    import geopandas as gpd
    from libpysal.weights import Queen, KNN
    from esda.moran import Moran
    import spreg

    shp_path = Path('/Users/nico/Documents/PhD research/energy poverty/Data/GDL Shapefiles V4/GDL Shapefiles V4.shp')
    gdf = gpd.read_file(shp_path, columns=['GDLcode', 'geometry'])
    sgdf = gdf.merge(model_data[['GDLcode', 'fe_residual'] + ['z_' + v for v in core_social + recommended_controls] + ['log_sufficiency_ratio']], on='GDLcode', how='inner')
    sgdf = sgdf.dropna(subset=['fe_residual', 'geometry']).copy()
    sgdf = sgdf.reset_index(drop=True)

    # Queen contiguity is the primary spatial weights definition. Islands are handled with KNN fallback for Moran sensitivity.
    wq = Queen.from_dataframe(sgdf, use_index=False)
    wq.transform = 'r'
    queen_islands = list(wq.islands)
    moran_q = Moran(sgdf['fe_residual'].values, wq, permutations=999, two_tailed=True)

    # KNN is used as a sensitivity because islands and non-contiguous island states are expected in global data.
    wk = KNN.from_dataframe(sgdf, k=6)
    wk.transform = 'r'
    moran_k = Moran(sgdf['fe_residual'].values, wk, permutations=999, two_tailed=True)

    spatial_rows = [
        {'weights': 'Queen contiguity', 'n_obs': len(sgdf), 'n_components_or_islands': len(queen_islands), 'moran_I': moran_q.I, 'expected_I': moran_q.EI, 'p_norm': moran_q.p_norm, 'p_sim': moran_q.p_sim},
        {'weights': 'KNN k=6', 'n_obs': len(sgdf), 'n_components_or_islands': 0, 'moran_I': moran_k.I, 'expected_I': moran_k.EI, 'p_norm': moran_k.p_norm, 'p_sim': moran_k.p_sim},
    ]
    spatial_table = pd.DataFrame(spatial_rows)
    spatial_table.to_csv(SI_DIR / 'Table_SI_RQ3_spatial_residual_morans_I.csv', index=False)
    spatial_table.to_csv(PROCESS_DIR / 'rq3_pretest_spatial_residual_morans_I.csv', index=False)

    # Moran scatterplot for SI.
    y = sgdf['fe_residual'].values
    wy = wk.sparse @ y
    fig, ax = plt.subplots(figsize=(5.5, 5), constrained_layout=True)
    ax.scatter(y, wy, s=18, alpha=.55, color='#4c78a8', edgecolor='none')
    slope, intercept = np.polyfit(y, wy, 1)
    xs = np.linspace(np.nanmin(y), np.nanmax(y), 100)
    ax.plot(xs, intercept + slope * xs, color='#d95f02', linewidth=1.5)
    ax.axhline(0, color='#777777', linewidth=.8)
    ax.axvline(0, color='#777777', linewidth=.8)
    ax.set_xlabel('Baseline country-FE residual')
    ax.set_ylabel('Spatially lagged residual (KNN k=6)')
    ax.set_title(f'Moran scatterplot, residuals (I = {moran_k.I:.3f}, p = {moran_k.p_sim:.3f})')
    fig.savefig(SI_DIR / 'Figure_SI_RQ3_spatial_residual_moran_scatter_knn6.png', bbox_inches='tight')
    plt.close(fig)

    # Spatial diagnostics from spreg OLS with country fixed effects dummies. This is diagnostic only.
    y_sp = sgdf[['log_sufficiency_ratio']].values
    X_base = sgdf[['z_' + v for v in core_social + recommended_controls]].copy()
    country_dummies = pd.get_dummies(model_data.set_index('GDLcode').loc[sgdf['GDLcode'], 'iso_code'], prefix='cty', drop_first=True).reset_index(drop=True)
    X_sp = pd.concat([X_base.reset_index(drop=True), country_dummies], axis=1).astype(float).values
    try:
        ols_sp = spreg.OLS(y_sp, X_sp, w=wk, spat_diag=True, moran=True, name_y='log_sufficiency_ratio')
        with open(PROCESS_DIR / 'rq3_pretest_spreg_ols_spatial_diagnostics_summary.txt', 'w') as f:
            f.write(str(ols_sp.summary))
    except Exception as e:
        with open(PROCESS_DIR / 'rq3_pretest_spreg_ols_spatial_diagnostics_error.txt', 'w') as f:
            f.write(repr(e))

    spatial_result = {
        'status': 'completed',
        'n_spatial_obs': int(len(sgdf)),
        'queen_islands': int(len(queen_islands)),
        'queen_moran_I': float(moran_q.I),
        'queen_p_sim': float(moran_q.p_sim),
        'knn6_moran_I': float(moran_k.I),
        'knn6_p_sim': float(moran_k.p_sim),
        'spatial_lag_or_error_branch_recommended': bool((moran_q.p_sim < 0.05) or (moran_k.p_sim < 0.05)),
    }
except Exception as e:
    spatial_result = {'status': 'failed', 'error': repr(e)}

with open(PROCESS_DIR / 'rq3_pretest_spatial_diagnostics_manifest.json', 'w') as f:
    json.dump(spatial_result, f, indent=2)
spatial_result


In [ ]:

# Compact machine-readable summary for the conversation and next modelling decisions.
summary = {
    'raw_rows': int(len(df_raw)),
    'complete_model_sample_rows': int(len(model_sample)),
    'complete_model_sample_countries': int(model_sample['iso_code'].nunique()),
    'complete_model_sample_regions': int(model_sample['analytical_region'].nunique()),
    'single_gdl_countries_in_model_sample': int(country_counts['single_gdl_country'].sum()),
    'max_vif_recommended_spec': float(recommended_vif['VIF'].max()),
    'recommended_spec_vif_ge_5': recommended_vif[recommended_vif['VIF'] >= 5][['variable','VIF']].to_dict('records'),
    'high_corr_abs_ge_0_65': high_corr[['var1','var2','pearson_r','p_value']].head(30).to_dict('records') if len(high_corr) else [],
    'spatial_diagnostics': spatial_result,
    'created_outputs': {
        'SI_dir': str(SI_DIR),
        'Process_dir': str(PROCESS_DIR),
    }
}
with open(PROCESS_DIR / 'rq3_pretest_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
summary
